In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def create_path(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:
root = os.getcwd()
root 

In [ ]:
dataset = "lfw"
sensitive = "gender"
taucc_path = root + f"/results/{dataset}/{sensitive}"
fatr_path = root + f"/algorithms/FATR/results/{dataset}/{sensitive}"
parity_path = root + f"/algorithms/C-Fairness-RecSys/reproducibility_study/Frisch_et_al/results/{dataset}/{sensitive}"

In [ ]:
alpha = 1.0
alpha_param = "fair_minority"

In [ ]:
taucc_fair_max = pd.read_csv(taucc_path + "/taucc_fair_max/init_random/aggregated.csv").query(f"{alpha_param} == {alpha}")
taucc_fair = pd.read_csv(taucc_path + "/taucc_fair/init_random/aggregated.csv").query(f"{alpha_param} == {alpha}")
taucc_vanilla = pd.read_csv(taucc_path + "/taucc_vanilla/init_random/aggregated.csv")
fatr = pd.read_csv(fatr_path + "/fair/aggregated.csv")
parity = pd.read_csv(parity_path + "/lbm_fair/aggregated_taus.csv")

In [ ]:
taucc_vanilla = taucc_vanilla.rename(columns={
    'ARI_mean':'ARI_true_labels_mean', 
    'ARI_std': 'ARI_true_labels_std',
    'ARI_var': 'ARI_true_labels_var',
    'AMI_mean':'AMI_true_labels_mean', 
    'AMI_std': 'AMI_true_labels_std',
    'AMI_var': 'AMI_true_labels_var',
    'NMI_mean':'NMI_true_labels_mean', 
    'NMI_std': 'NMI_true_labels_std',
    'NMI_var': 'NMI_true_labels_var'
})

In [ ]:
plot_path = root + f"/plots/{dataset}/{sensitive}/{alpha_param}_all"
create_path(plot_path)
plot_path

## Plot of Results

In [ ]:
if alpha_param == "fair_majority":
    x = np.array(taucc_fair_max["fair_minority"].values)
    print("from fair minority")
    print(x)
else:
    x = np.array(taucc_fair_max["fair_majority"].values)
    print("from fair majority")
    print(x)

In [ ]:
def plot_metric(
    x,
    taucc_vanilla,
    taucc_fair,
    taucc_fair_max,
    metric,
    alpha,
    dataset,
    plot_path,
    metric_label,
    title,
    range_values=None,
    fatr=None,
    parity=None
):
    # Colori palette Wong (2011) - colorblind-friendly
    COLOR_VANILLA  = "#E69F00"
    COLOR_FAIR     = "#0072B2"
    COLOR_FAIR_MAX = "#009E73"
    COLOR_PARITY   = "#D55E00"
    COLOR_FATR     = "#CC79A7"

    mean_col = f"{metric}_mean"
    std_col  = f"{metric}_std"

    cols = [mean_col, std_col]
    if all(c in taucc_vanilla.columns for c in cols):
        include_vanilla=True
    else:
        include_vanilla=False
        
    if include_vanilla:
        # Fast TauCC
        vanilla_mean = np.full(len(x), taucc_vanilla[mean_col].values[0])
        vanilla_std  = np.full(len(x), taucc_vanilla[std_col].values[0])

    # Fair TauCC v1
    fair_mean = np.array(taucc_fair[mean_col].values)
    fair_std  = np.array(taucc_fair[std_col].values)

    # Fair TauCC v2 (max)
    fair_max_mean = np.array(taucc_fair_max[mean_col].values)
    fair_max_std  = np.array(taucc_fair_max[std_col].values)
    
    if parity is not None:
        parity_mean = np.full(len(x), parity[mean_col].values[0])
        parity_std  = np.full(len(x), parity[std_col].values[0])
    if fatr is not None:
        fatr_mean = np.full(len(x), fatr[mean_col].values[0])
        fatr_std  = np.full(len(x), fatr[std_col].values[0])
    

    # Plot
    fig, ax = plt.subplots()

    if include_vanilla:
        ax.plot(x, vanilla_mean, label="Fast $\\tau$CC", color=COLOR_VANILLA, linestyle='--', linewidth=1.5)
        ax.fill_between(x, vanilla_mean - vanilla_std, vanilla_mean + vanilla_std, alpha=0.15, color=COLOR_VANILLA)

    ax.plot(x, fair_mean, label="Fair $\\tau$CC v1", color=COLOR_FAIR, linewidth=1.5)
    ax.fill_between(x, fair_mean - fair_std, fair_mean + fair_std, alpha=0.15, color=COLOR_FAIR)

    ax.plot(x, fair_max_mean, label="Fair $\\tau$CC v2", color=COLOR_FAIR_MAX, linewidth=1.5)
    ax.fill_between(x, fair_max_mean - fair_max_std, fair_max_mean + fair_max_std, alpha=0.15, color=COLOR_FAIR_MAX)
    
    if parity is not None:
        ax.plot(x, parity_mean, label="Parity LBM", color=COLOR_PARITY, linestyle='dotted', linewidth=1.5)
        ax.fill_between(x, parity_mean - parity_std, parity_mean + parity_std, alpha=0.15, color=COLOR_PARITY)
    if fatr is not None:
        ax.plot(x, fatr_mean, label="FATR", color=COLOR_FATR, linestyle='dotted', linewidth=1.5)
        ax.fill_between(x, fatr_mean - fatr_std, fatr_mean + fatr_std, alpha=0.15, color=COLOR_FATR)
    
    if range_values is not None:
        ax.set_ylim(range_values[0], range_values[1])
    
    ax.set_xlim(0.0, 1.0)
    ax.legend(framealpha=0.9, edgecolor='gray', fontsize=10)
    
    if alpha_param == "fair_majority":
        ax.set_xlabel('$\\alpha$ minority group')
        ax.set_title(f"{title} \n $\\alpha$ majority group = {alpha}")
    else:
        ax.set_xlabel('$\\alpha$ majority group')
        ax.set_title(f"{title} \n $\\alpha$ minority group = {alpha}")
    
    ax.set_ylabel(metric_label)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

    plt.tight_layout()
    plt.savefig(f"{plot_path}/{metric}.png", dpi=300)
    #plt.show()
    plt.close(fig)

In [ ]:
metrics = [
    #("balance_bera",     "balance", "Balance", [0.0,1.0]),
    #("tau_x",            "tau x",   "tau x", None),
    #("tau_y",            "tau y",   "tau y", None),
    #("ARI_true_labels",  "ARI",     "ARI w.r.t. true labels", None),
    #("ARI_rows",         "ARI",     "ARI w.r.t. row clusters", None),
    #("ARI_cols",         "ARI",     "ARI w.r.t. column clusters", None),
    ("time", "time", "Time (in seconds)", None)
]

for metric, metric_label, title, range_values in metrics:    
    plot_metric(
        x=x,
        taucc_vanilla=taucc_vanilla,
        taucc_fair=taucc_fair,
        taucc_fair_max=taucc_fair_max,
        parity=parity,
        fatr=fatr,
        metric=metric,
        metric_label=metric_label,
        title=title,
        range_values=range_values,
        alpha=alpha,
        dataset=dataset,
        plot_path=plot_path,
    )

# Color map

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
def create_path(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [3]:
root = os.getcwd()
root 

'/home/peiretti/fair-clustering'

In [4]:
dataset = "yelp"
sensitive = "gender"
taucc_path = root + f"/results/{dataset}/{sensitive}"

In [5]:
taucc_fair_max = pd.read_csv(taucc_path + "/taucc_fair_max/init_random/aggregated.csv")
taucc_fair = pd.read_csv(taucc_path + "/taucc_fair/init_random/aggregated.csv")
taucc_vanilla = pd.read_csv(taucc_path + "/taucc_vanilla/init_random/aggregated.csv")

taucc_vanilla = taucc_vanilla.rename(columns={
    'ARI_mean':'ARI_true_labels_mean', 
    'ARI_std': 'ARI_true_labels_std',
    'ARI_var': 'ARI_true_labels_var',
    'AMI_mean':'AMI_true_labels_mean', 
    'AMI_std': 'AMI_true_labels_std',
    'AMI_var': 'AMI_true_labels_var',
    'NMI_mean':'NMI_true_labels_mean', 
    'NMI_std': 'NMI_true_labels_std',
    'NMI_var': 'NMI_true_labels_var'
})

In [6]:
heatmap_path = root + f"/plots/{dataset}/{sensitive}/heatmap"
#create_path(heatmap_path)
heatmap_path

'/home/peiretti/fair-clustering/plots/yelp/gender/heatmap'

In [11]:
def plot_heatmap(
    df,
    metric,
    algorithm,
    dataset,
    sensitive,
    plot_path,
    range_values=None,
    metric_label=None,
    title=None,
    col_param="fair_majority",
    row_param="fair_minority"
):
    
    if algorithm == "taucc_fair_max":
        algorithm_name = "Fair $\\tau$CC v2"
    elif algorithm == "taucc_fair":
        algorithm_name = "Fair $\\tau$CC v1"
    else:
        raise Exception("The algorithm does not exist. Choose an option: taucc_fair or taucc_fair_max.")
    
    mean_col = f"{metric}_mean"

    # Pivot: righe = fair_minority, colonne = fair_majority
    pivot = df.pivot(index=row_param, columns=col_param, values=mean_col)

    fig, ax = plt.subplots(figsize=(6, 5))
    
    if range_values is not None:
        vmin=range_values[0]
        vmax=range_values[1]
    else:
        vmin = pivot.values.min()
        vmax = pivot.values.max()

    pivot = pivot.sort_index(ascending=False)
    
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        annot=False,
        cbar_kws={"label": metric_label if metric_label else metric}
    )
    
    ax.set_xlabel(r"$\alpha$ majority",fontsize=16)
    ax.set_ylabel(r"$\alpha$ minority",fontsize=16)
    ax.set_title(f"{dataset} - {sensitive} \n {algorithm_name} - {title}", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    
    cbar = ax.collections[0].colorbar
    cbar.set_label(metric_label, fontsize=16)
    cbar.ax.tick_params(labelsize=14)
    
    ax.grid(False)

    plt.tight_layout()
    plt.savefig(f"{plot_path}/{algorithm}_{metric}.png", dpi=300, bbox_inches='tight')
    #plt.show()
    plt.close(fig)

In [12]:
metrics = [
    ("balance_bera",     "balance", "Balance", [0.0,1.0]),
    #("tau_x",            "tau x",   "tau x", None),
    #("tau_y",            "tau y",   "tau y", None),
    #("ARI_true_labels",  "ARI",     "ARI w.r.t. true labels", None),
    #("ARI_rows",         "ARI",     "ARI w.r.t. row clusters", None),
    #("ARI_cols",         "ARI",     "ARI w.r.t. column clusters", None),
]

for metric, metric_label, title, range_values in metrics:
    
    plot_heatmap(
        df=taucc_fair,
        metric=metric,
        algorithm="taucc_fair",
        dataset=dataset,
        sensitive=sensitive,
        plot_path=heatmap_path,
        metric_label=metric_label,
        title=title,
        range_values=range_values
    )

    plot_heatmap(
        df=taucc_fair_max,
        metric=metric,
        algorithm="taucc_fair_max",
        dataset=dataset,
        sensitive=sensitive,
        plot_path=heatmap_path,
        metric_label=metric_label,
        title=title,
        range_values=range_values
    )

# Heatmap FairTauCC v1 e v2 con alpha minority2 = 1.0

In [ ]:
def plot_heatmap(
    df_v1,
    df_v2,
    metric,
    dataset,
    sensitive,
    plot_path,
    range_values=None,
    metric_label=None,
    title=None,
    col_param="fair_majority",
    row_param="fair_minority",
    alpha_value=1.0
):
    
    configs = [
        (df_v1, "taucc_fair",     "Fair $\\tau$CC v1"),
        (df_v2, "taucc_fair_max", "Fair $\\tau$CC v2"),
    ]

    mean_col = f"{metric}_mean"

    for df, algo_key, algo_name in configs:
        # Filtra alpha_minority2 == 1.0
        if "fair_minority2" not in df.columns:
            raise ValueError(f"Colonna 'fair_minority2' non trovata nel dataframe di {algo_name}.")
        df_filtered = df[df["fair_minority2"] == alpha_value].copy()

        pivot = df_filtered.pivot(index=row_param, columns=col_param, values=mean_col)
        pivot = pivot.sort_index(ascending=False)

        if range_values is not None:
            vmin, vmax = range_values
        else:
            vmin = pivot.values.min()
            vmax = pivot.values.max()

        fig, ax = plt.subplots(figsize=(6, 5))

        sns.heatmap(
            pivot,
            ax=ax,
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            annot=False,
            cbar_kws={"label": metric_label if metric_label else metric},
        )

        ax.set_xlabel(r"$\alpha$ majority",fontsize=16)
        ax.set_ylabel(r"$\alpha$ minority",fontsize=16)
        ax.set_title(f"{algo_name} - {title}\n($\\alpha$ minority2={alpha_value})", fontsize=16)
        
        ax.tick_params(axis="both", labelsize=14)
    
        cbar = ax.collections[0].colorbar
        cbar.set_label(metric_label, fontsize=16)
        cbar.ax.tick_params(labelsize=14)
        
        ax.grid(False)
        plt.tight_layout()

        out_path = f"{plot_path}/{algo_key}_{metric}_alphamin2_{alpha_value}.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        #plt.show()
        plt.close(fig)
        print(f"Salvato: {out_path}")

In [ ]:
metrics = [
    ("balance_bera",     "balance", "Balance", [0.0,1.0]),
    ("tau_x",            "tau x",   "tau x", None),
    ("tau_y",            "tau y",   "tau y", None),
    #("ARI_true_labels",  "ARI",     "ARI w.r.t. true labels", None),
    #("ARI_rows",         "ARI",     "ARI w.r.t. row clusters", None),
    #("ARI_cols",         "ARI",     "ARI w.r.t. column clusters", None),
]


for metric, metric_label, title, range_v in metrics:
    plot_heatmap(
        df_v1=taucc_fair,
        df_v2=taucc_fair_max,
        metric=metric,
        metric_label=metric_label,
        range_values=range_v,
        dataset=dataset,
        sensitive=sensitive,
        plot_path=heatmap_path,
        title=title,
        alpha_value=1.0
    )

# Old code

Balance

In [ ]:
# Colori adatti per paper (palette colorblind-friendly)
COLOR_VANILLA   = "#E69F00"  # arancione
COLOR_FAIR      = "#0072B2"  # blu scuro
COLOR_FAIR_MAX  = "#009E73"  # verde teal

# Vanilla TauCC (fisso, retta orizzontale)
vanilla_bera_mean = np.full(len(x), taucc_vanilla["balance_bera_mean"].values[0])
vanilla_bera_std  = np.full(len(x), taucc_vanilla["balance_bera_std"].values[0])

# Fair TauCC
balance_bera_mean_fair     = np.array(taucc_fair["balance_bera_mean"].values)
balance_bera_std_fair      = np.array(taucc_fair["balance_bera_std"].values)

# Fair Max TauCC
balance_bera_mean_fair_max = np.array(taucc_fair_max["balance_bera_mean"].values)
balance_bera_std_fair_max  = np.array(taucc_fair_max["balance_bera_std"].values)

plt.plot(x, vanilla_bera_mean, label='Fast $\\tau$CC', color=COLOR_VANILLA, linestyle='--', linewidth=1.5)
plt.fill_between(x, vanilla_bera_mean - vanilla_bera_std, vanilla_bera_mean + vanilla_bera_std, alpha=0.15, color=COLOR_VANILLA)

plt.plot(x, balance_bera_mean_fair, label='Fair $\\tau$CC v1', color=COLOR_FAIR, linewidth=1.5)
plt.fill_between(x, balance_bera_mean_fair - balance_bera_std_fair, balance_bera_mean_fair + balance_bera_std_fair, alpha=0.15, color=COLOR_FAIR)

plt.plot(x, balance_bera_mean_fair_max, label='Fair $\\tau$CC v2', color=COLOR_FAIR_MAX, linewidth=1.5)
plt.fill_between(x, balance_bera_mean_fair_max - balance_bera_std_fair_max, balance_bera_mean_fair_max + balance_bera_std_fair_max, alpha=0.15, color=COLOR_FAIR_MAX)

plt.ylim(0.0, 1.0)
plt.xlim(0.0, 1.0)
plt.legend(framealpha=0.9, edgecolor='gray', fontsize=10)
plt.xlabel('$\\alpha$ minority group')
plt.ylabel('balance')
plt.title(f'Balance \n $\\alpha$ majority group = {alpha}')
plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
plt.tight_layout()
#plt.savefig(plot_path + f"/{dataset}_fairness.png", dpi=300)
plt.show()

In [ ]:
"""
# Vanilla TauCC
vanilla_bera_mean = np.full(len(x), df_vanilla["balance_bera_mean"].values[0])
vanilla_bera_std = np.full(len(x), df_vanilla["balance_bera_std"].values[0])

# Fair TauCC
balance_bera_mean = np.array(df["balance_bera_mean"].values)
balance_bera_std = np.array(df["balance_bera_std"].values)

plt.plot(x, balance_bera_mean, label='fair')
plt.fill_between(x, balance_bera_mean - balance_bera_std, balance_bera_mean + balance_bera_std, alpha=0.2, color='g')
plt.plot(x, vanilla_bera_mean, label='vanilla')
plt.fill_between(x, vanilla_bera_mean - vanilla_bera_std, vanilla_bera_mean + vanilla_bera_std, alpha=0.2)
plt.ylim(0.0, 1.0)
plt.xlim(0.0, 1.0)
plt.legend()
plt.xlabel('$\\alpha$ minority group')
plt.ylabel('balance')
plt.title(f'Balance [Bera et al.] \n $\\alpha$ majority group = {alpha}')
plt.grid(True)
plt.tight_layout()
#plt.show()
plt.savefig(plot_path + f"/{dataset}_fairness.png", dpi=300)
"""

ARI w.r.t. true labels

In [ ]:
vanilla_ARI_mean = np.full(len(x), df_vanilla["ARI_mean"].values[0])
vanilla_ARI_std = np.full(len(x), df_vanilla["ARI_std"].values[0])

ARI_mean = np.array(df["ARI_true_labels_mean"].values)
ARI_std = np.array(df["ARI_true_labels_std"].values)

plt.plot(x, ARI_mean, label='fair')
plt.fill_between(x, ARI_mean - ARI_std, ARI_mean + ARI_std, alpha=0.2, color='g')
plt.plot(x, vanilla_ARI_mean, label='vanilla')
plt.fill_between(x, vanilla_ARI_mean - vanilla_ARI_std, vanilla_ARI_mean + vanilla_ARI_std, alpha=0.2)
plt.ylim(0.0, 0.15)
plt.xlim(0.0, 1.0)
plt.legend()
plt.xlabel('$\\alpha$ minority group')
plt.ylabel('ARI')
plt.title(f'ARI w.r.t. true labels \n $\\alpha$ majority group = {alpha}')
plt.grid(True)
plt.tight_layout()
#plt.show()
plt.savefig(plot_path + f"/{dataset}_ARI_true.png", dpi=300)

ARI w.r.t. vanilla TauCC

In [ ]:
ARIrows_mean = np.array(df["ARI_rows_mean"].values)
ARIrows_std = np.array(df["ARI_rows_std"].values)
ARI_mean = np.array(df["ARI_cols_mean"].values)
ARI_std = np.array(df["ARI_cols_std"].values)

plt.plot(x, ARIrows_mean, label='ARI w.r.t. rows', color="green")
plt.fill_between(x, ARIrows_mean - ARIrows_std, ARIrows_mean + ARIrows_std, alpha=0.2, color="green")
plt.plot(x, ARI_mean, label='ARI w.r.t. cols')
plt.fill_between(x, ARI_mean - ARI_std, ARI_mean + ARI_std, alpha=0.2)
plt.ylim(-1.0, 1.0)
plt.xlim(0.0, 1.0)
plt.legend()
plt.xlabel('$\\alpha$ minority group')
plt.ylabel('ARI')
plt.title(f'ARI w.r.t. $\\tau$CC rows and columns \n $\\alpha$ majority group = {alpha}')
plt.grid(True)
plt.tight_layout()
#plt.show()
plt.savefig(plot_path + f"/{dataset}_ARI.png", dpi=300)

tau x and tau y

In [ ]:
vanilla_taux_mean = np.full(len(x), df_vanilla["tau_x_mean"].values[0])
vanilla_taux_std = np.full(len(x), df_vanilla["tau_x_std"].values[0])
vanilla_tauy_mean = np.full(len(x), df_vanilla["tau_y_mean"].values[0])
vanilla_tauy_std = np.full(len(x), df_vanilla["tau_y_std"].values[0])

taux_mean = np.array(df["tau_x_mean"].values)
taux_std = np.array(df["tau_x_std"].values)
tauy_mean = np.array(df["tau_y_mean"].values)
tauy_std = np.array(df["tau_y_std"].values)

In [ ]:
# fair
plt.plot(x, taux_mean, label='tau x (fair)')
plt.fill_between(x, taux_mean - taux_std, taux_mean + taux_std, alpha=0.2)
plt.plot(x, tauy_mean, label='tau y (fair)')
plt.fill_between(x, tauy_mean - tauy_std, tauy_mean + tauy_std, alpha=0.2)

# vanilla
plt.plot(x, vanilla_taux_mean, label='tau x (vanilla)')
plt.fill_between(x, vanilla_taux_mean - vanilla_taux_std, vanilla_taux_mean + vanilla_taux_std, alpha=0.2)
plt.plot(x, vanilla_tauy_mean, label='tau y (vanilla)')
plt.fill_between(x, vanilla_tauy_mean - vanilla_tauy_std, vanilla_tauy_mean + vanilla_tauy_std, alpha=0.2) 

plt.legend()
plt.xlabel('$\\alpha$ minority group')
plt.ylabel('tau_x, tau_y')
plt.title(f'tau_x and tau_y \n $\\alpha$ majority group = {alpha}')

plt.xlim(0.0, 1.0)

plt.grid(True)
plt.tight_layout()
#plt.show()
plt.savefig(plot_path + f"/{dataset}_tau.png", dpi=300)